# Lab | Chains in LangChain

## Outline

* LLMChain
* Sequential Chains
  * SimpleSequentialChain
  * SequentialChain
* Router Chain

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
HUGGINGFACEHUB_API_TOKEN = os.getenv('HUGGINGFACEHUB_API_TOKEN')


In [3]:
# Check whether Python successfully loaded your OpenAI API key.
# bool() returns True if the key is present and nonempty.
# This avoids displaying the secret key itself.

print("OpenAI API key found:", bool(OPENAI_API_KEY))


OpenAI API key found: True


In [5]:
pip install pandas

  Using cached numpy-2.5.4-cp312-cp312-macosx_14_0_arm64.whl.metadata (6.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 1.6 MB/s  0:00:07 eta 0:00:01
Using cached numpy-2.5.4-cp312-cp312-macosx_14_0_arm64.whl (5.5 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.2.19 requires numpy<2.0.0,>=1.26.0; python_version >= "3.12", but you have numpy 2.5.4 which is incompatible.
langchain 0.2.17 requires numpy<2.0.0,>=1.26.0; python_version >= "3.12", but you have numpy 2.5.4 which is incompatible.

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
No

In [6]:
import pandas as pd
df = pd.read_csv('./data/Data.csv')

In [7]:
df.head()

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld,I loved this product. But they only seem to la...


## LLMChain

In [8]:
pip install langchain_community

  Using cached numpy-1.26.4-cp312-cp312-macosx_11_0_arm64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-macosx_11_0_arm64.whl (13.7 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.5.4
    Uninstalling numpy-2.5.4:
      Successfully uninstalled numpy-2.5.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pandas 3.1.0 requires numpy>=2.0.2; python_version < "3.14", but you have numpy 1.26.4 which is incompatible.

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.chains import LLMChain

In [14]:
# Choose an OpenAI model explicitly instead of relying on the old default.
# Keep our selected temperature of 0.3 for consistent product descriptions.

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0.3
)

Temperature controls how much randomness is introduced when a model selects its next tokens. Lower values generally produce more consistent outputs, while higher values allow more variation. The exact effect and supported range depend on the model.

In [15]:
# Create a reusable prompt for describing products.
# {product} will be replaced with the product we choose.

prompt = ChatPromptTemplate.from_template(
    "Write a clear, professional description of {product} "
    "in no more than three sentences."
)

In [16]:

chain = LLMChain(llm=llm, prompt=prompt)

In [17]:
# Store the name of the product in a string variable.
product = "MacBook Pro"

# Run the chain using the product as its input.
# The result is the text generated by the language model.
chain.run(product)

'The MacBook Pro is a high-performance laptop designed by Apple, featuring advanced processors and a vibrant Retina display for exceptional visual clarity. It offers powerful graphics, long battery life, and a sleek, durable aluminum chassis, making it ideal for professionals and creatives. With macOS integration and a range of connectivity options, the MacBook Pro delivers a seamless and efficient user experience.'

## SimpleSequentialChain

In [18]:
from langchain.chains import SimpleSequentialChain

In [22]:
# Configure the model explicitly.
# Temperature 0.9 allows more variation for creative marketing content.
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0.9
)

# Create a template for the first chain.
# {product} will receive the product name.
first_prompt = ChatPromptTemplate.from_template(
    "Write a short, engaging description of {product}."
)

# Connect the model to the first prompt.
chain_one = LLMChain(llm=llm, prompt=first_prompt)

# Prompt template 2:
# Take the description generated by the first chain
# and use it to create a marketing slogan.

second_prompt = ChatPromptTemplate.from_template(
    "Create a short, catchy marketing slogan based on this product description: {description}"
)

# Create the second chain using the same language model.
chain_two = LLMChain(llm=llm, prompt=second_prompt)

In [23]:
overall_simple_chain = SimpleSequentialChain(chains=[chain_one, chain_two],
                                             verbose=True
                                            )

In [24]:
overall_simple_chain.run(product)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


The MacBook Pro is Apple’s powerhouse laptop, blending sleek design with cutting-edge performance. Featuring a vibrant Retina display, blazing-fast processors, and all-day battery life, it’s built for creatives, professionals, and anyone who demands speed and precision on the go. Whether editing videos, coding, or multitasking, the MacBook Pro delivers smooth, reliable power wrapped in a stunning, portable package.
MacBook Pro: Power, Precision, Perfected.

> Finished chain.


'MacBook Pro: Power, Precision, Perfected.'

**Repeat the above twice for different products**

## SequentialChain

In [25]:
from langchain.chains import SequentialChain

In [28]:
# Use a model that supports the lab's API calls.
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0.3
)

# Translate the original review into Spanish.
first_prompt = ChatPromptTemplate.from_template(
    "Translate this customer review into Spanish: {Review}"
)

# Name the translated output so later chains can use it.
chain_one = LLMChain(
    llm=llm,
    prompt=first_prompt,
    output_key="Spanish_Review"
)

In [29]:
# Summarize the Spanish review produced by Chain 1.
second_prompt = ChatPromptTemplate.from_template(
    "Summarize this customer review in one sentence: {Spanish_Review}"
)

# Store the generated summary under the name "summary".
chain_two = LLMChain(
    llm=llm,
    prompt=second_prompt,
    output_key="summary"
)

In [31]:
# Create a prompt to translate the summary into English.
# {summary} receives the output produced by Chain 2.
third_prompt = ChatPromptTemplate.from_template(
    "Translate the following Spanish summary into English: {summary}"
)

# Create Chain 3 and name its output "English_summary".
chain_three = LLMChain(
    llm=llm,
    prompt=third_prompt,
    output_key="English_summary"
)

In [32]:
# Create a prompt that uses two previous variables:
# Review: the original customer review.
# English_summary: the summary translated back into English.
fourth_prompt = ChatPromptTemplate.from_template(
    "You are a professional customer service representative. "
    "Write a friendly and helpful follow-up message to the customer. "
    "Original review: {Review}\n"
    "Review summary: {English_summary}\n"
    "Acknowledge the feedback and avoid inventing company policies."
)

# Create the fourth chain and name its output.
chain_four = LLMChain(
    llm=llm,
    prompt=fourth_prompt,
    output_key="followup_message"
)

In [34]:
# Connect the four individual chains into one workflow.
overall_chain = SequentialChain(
    # Run the chains in this order.
    chains=[chain_one, chain_two, chain_three, chain_four],

    # The original customer review starts the workflow.
    input_variables=["Review"],

    # Select which generated results we want returned.
    output_variables=[
        "Spanish_Review",
        "summary",
        "English_summary",
        "followup_message"
    ],

    # Display information about the execution.
    verbose=False
)

In [35]:
review = df.Review[5]
overall_chain(review)

/var/folders/1g/3xm4lnvn64z02xj5s9vlj7hc0000gn/T/ipykernel_9951/1992003631.py:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use invoke instead.
  overall_chain(review)


{'Review': "Je trouve le goût médiocre. La mousse ne tient pas, c'est bizarre. J'achète les mêmes dans le commerce et le goût est bien meilleur...\nVieux lot ou contrefaçon !?",
 'Spanish_Review': 'Encuentro el sabor mediocre. La espuma no se mantiene, es raro. Compro los mismos en el comercio y el sabor es mucho mejor...\n¿Lote viejo o falsificación?',
 'summary': 'The customer finds the flavor mediocre, the foam unstable, and suspects the product might be old stock or counterfeit compared to better-tasting ones bought elsewhere.',
 'English_summary': 'The customer finds the flavor mediocre, the foam unstable, and suspects the product might be old stock or counterfeit compared to better-tasting ones purchased elsewhere.',
 'followup_message': 'Bonjour,\n\nMerci beaucoup d’avoir pris le temps de partager votre ressenti avec nous. Nous sommes désolés d’apprendre que le goût et la texture de notre produit n’ont pas été à la hauteur de vos attentes. Votre remarque concernant la mousse et 

**Repeat the above twice for different products or reviews**

## Router Chain

In [48]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts, 
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity. 

Here is a question:
{input}"""

biology_template = """You are an excellent biologist. \
You have a deep understanding of living organisms, \
from the molecular and cellular level to entire ecosystems. \
You are skilled at observing patterns in nature, analyzing biological data, \
and explaining complex processes like evolution, genetics, physiology, and ecology. \
You can clearly communicate how life functions and adapts, \
and you make connections between different biological concepts \
to answer challenging questions.

Here is a question:
{input}"""

In [49]:
prompt_infos = [
    {
        "name": "physics", 
        "description": "Good for answering questions about physics", 
        "prompt_template": physics_template
    },
    {
        "name": "math", 
        "description": "Good for answering math questions", 
        "prompt_template": math_template
    },
    {
        "name": "History", 
        "description": "Good for answering history questions", 
        "prompt_template": history_template
    },
    {
        "name": "computer science", 
        "description": "Good for answering computer science questions", 
        "prompt_template": computerscience_template
    },
    {
        "name": "biology",
        "description": "Good for answering biology questions",
        "prompt_template": biology_template
    }
]

In [50]:
from langchain.chains.router import MultiPromptChain
from langchain.chains.router.llm_router import LLMRouterChain,RouterOutputParser
from langchain.prompts import PromptTemplate

In [51]:
# Explicitly select the model that worked in our earlier exercises.
# Temperature 0 minimizes sampling randomness for routing decisions.
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

# Verify which model is configured.
print("Router model:", llm.model_name)

Router model: gpt-4.1-mini


In [52]:
# Check the model used by the router itself.
print("Router:", router_chain.llm_chain.llm.model_name)

# Check the model used by the physics destination chain.
print("Physics:", destination_chains["physics"].llm.model_name)

Router: gpt-3.5-turbo
Physics: gpt-3.5-turbo


In [53]:
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain  
    
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

In [54]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)

In [55]:
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ name of the prompt to use or "DEFAULT"
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: "destination" MUST be one of the candidate prompt \
names specified below OR it can be "DEFAULT" if the input is not\
well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

In [56]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

router_chain = LLMRouterChain.from_llm(llm, router_prompt)

In [57]:
chain = MultiPromptChain(router_chain=router_chain, 
                         destination_chains=destination_chains, 
                         default_chain=default_chain, verbose=True
                        )

In [58]:
chain.run("What is black body radiation?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


physics: {'input': 'What is black body radiation?'}
> Finished chain.


"Black body radiation is the electromagnetic radiation emitted by an idealized object called a black body, which absorbs all incident radiation regardless of frequency or angle. When this object is at a certain temperature, it emits radiation with a characteristic spectrum that depends only on its temperature, not on its material. This spectrum is continuous and peaks at a wavelength inversely related to the temperature (Wien's law). Black body radiation was crucial in the development of quantum mechanics because classical physics couldn't explain its observed spectrum, leading Planck to propose quantized energy levels."

In [59]:
chain.run("what is 2 + 2")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


math: {'input': 'what is 2 + 2'}
> Finished chain.


"Great! Let's break down the problem:\n\n1. Identify the numbers involved: 2 and 2.\n2. Understand the operation: addition (+).\n3. Add the two numbers together: 2 + 2.\n\nNow, performing the addition:\n\n2 + 2 = 4.\n\nSo, the answer is **4**."

In [60]:
chain.run("Why does every cell in our body contain DNA?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


biology: {'input': 'Why does every cell in our body contain DNA?'}
> Finished chain.


'Every cell in our body contains DNA because DNA holds the complete set of genetic instructions necessary for the development, functioning, growth, and reproduction of the organism. Here’s a detailed explanation:\n\n1. **Genetic Blueprint:** DNA (deoxyribonucleic acid) contains genes, which are sequences of nucleotides that encode the information to produce proteins and RNA molecules. Proteins carry out most cellular functions, from structural roles to enzymatic activities.\n\n2. **Cellular Identity and Function:** Although different cell types (e.g., muscle cells, nerve cells, skin cells) perform specialized functions, they all originate from a single fertilized egg and share the same DNA. The differences arise because different genes are expressed (turned on or off) in different cells, but the underlying genetic code is the same.\n\n3. **Growth and Development:** During development, cells divide repeatedly. Each new cell must have a complete copy of the DNA to maintain the organism’s

**Repeat the above at least once for different inputs and chains executions - Be creative!**

In [61]:
# Test whether the router recognizes a computer science question.
# We expect it to select the computer science destination.

result = chain.run(
    "What is the difference between a Python list and a tuple?"
)

# Display the answer returned by the selected chain.
print(result)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


computer science: {'input': 'What is the difference between a Python list and a tuple?'}
> Finished chain.
Great question! Understanding the difference between a Python list and a tuple is fundamental for writing efficient and clear Python code. Here's a clear, step-by-step explanation highlighting their key differences:

### 1. **Mutability**
- **List:** Mutable — you can modify a list after its creation (e.g., add, remove, or change elements).
- **Tuple:** Immutable — once a tuple is created, you cannot change its elements.

### 2. **Syntax**
- **List:** Defined using square brackets `[]`.
  ```python
  my_list = [1, 2, 3]
  ```
- **Tuple:** Defined using parentheses `()` or just commas.
  ```python
  my_tuple = (1, 2, 3)
  # or
  my_tuple = 1, 2, 3
  ```

### 3. **Use Cases**
- **List:** Use when you need a collection of items that may change during program execution.
- **Tuple:** Use when you want a fixed collection of items, ensuring data integrity and potentially improving perfor